# Notebook 00 — scaffold smoke test (Phase 1)

**Experiment:** EXP-0001d · **Phase:** 1 · **Date:** 2026-11-08

**Purpose** — prove the environment end-to-end: package import, config
fingerprint, Kathmandu offset (DC4), seed policy (DC14), DuckDB round-trip,
and the real NABIL 2024-12 sample loaded into `floorsheet_raw` with the
brief's exact check query.

**Prerequisites:** `make bootstrap install test-fast smoke` executed.

**Expected outputs:** version table, config sha256, offset check, DuckDB
query rows, HHI/self-loop sample table (EXP-0001c), STATUS REPORT cell.

**Status markers (R4/R5):** all numbers below are [REAL] — derived from the
local socrateai archive files (MIT), sha256-tracked.


In [1]:

import sys, importlib
print(f"python {sys.version.split()[0]}")
mods = {"duckdb": "duckdb", "pandas": "pandas", "numpy": "numpy",
        "networkx": "networkx", "sklearn": "scikit-learn",
        "linearmodels": "linearmodels", "statsmodels": "statsmodels",
        "plotly": "plotly"}
for imp in ("duckdb", "pandas", "numpy", "networkx", "sklearn",
            "linearmodels", "statsmodels", "plotly"):
    m = importlib.import_module(imp)
    print(f"{imp:14s} {getattr(m, '__version__', '?')}")
assert sys.version_info[:2] == (3, 11), "brief pins python 3.11"


python 3.11.16
duckdb         2.0.0


pandas         3.1.1
numpy          2.4.6
networkx       3.6.1


sklearn        1.9.1
linearmodels   7.1
statsmodels    0.15.0
plotly         7.1.0


In [2]:

from nepsemsc.utils import config
cfg = config.load_config()
print("config sha256:", cfg["_fingerprint_sha256"][:16], "…")
assert cfg["_fingerprint_sha256"]

from nepsemsc.utils.tz import now_ktm, verify_offset
print("Kathmandu offset:", now_ktm().utcoffset(), "| DC4 check:", verify_offset())
assert verify_offset()

from nepsemsc.utils.seeds import IF_FOLD_SEEDS, PLACEBO_SEEDS
print("IF fold seeds:", IF_FOLD_SEEDS, "| #placebo seeds:", len(PLACEBO_SEEDS))
assert len(PLACEBO_SEEDS) == 200


config sha256: 29a43e8c12c26260 …
Kathmandu offset: 5:45:00 | DC4 check: True
IF fold seeds: [917501, 917502, 917503, 917504] | #placebo seeds: 200


In [3]:

from pathlib import Path
import duckdb
from nepsemsc.data.duckdb_loader import DB_PATH, DAILY_COUNTS_SQL, get_conn, init_db
assert Path(DB_PATH).exists(), "DuckDB missing — run scripts/init_duckdb_load_sample.py"
con = get_conn()
init_db(con)
total = con.execute("SELECT COUNT(*) FROM floorsheet_raw").fetchone()[0]
print(f"floorsheet_raw rows: {total:,}  [REAL — NABIL 2024-12 extract of socrateai archive]")
table = con.execute("SELECT * FROM floorsheet_raw LIMIT 5").df()
display(table)
rows = con.execute(DAILY_COUNTS_SQL).fetchall()
print(f"DAILY_COUNTS_SQL (brief query): {len(rows)} rows")
for r in rows[:10]:
    print(r)
con.close()


floorsheet_raw rows: 13,099  [REAL — NABIL 2024-12 extract of socrateai archive]


,date,contract_no,symbol,price,volume,amount,buyer_broker_id,seller_broker_id
0,2024-12-01,2024120101019538,NABIL,536.0,69,36984.0,52,16
1,2024-12-01,2024120101019423,NABIL,536.0,231,123816.0,01,16
2,2024-12-01,2024120101019422,NABIL,536.0,150,80400.0,01,58
3,2024-12-01,2024120101019421,NABIL,536.0,69,36984.0,01,45
4,2024-12-01,2024120101019346,NABIL,536.0,40,21440.0,49,45


DAILY_COUNTS_SQL (brief query): 10 rows
('NABIL', datetime.date(2024, 12, 1), 695, 58, 55)
('NABIL', datetime.date(2024, 12, 2), 776, 59, 60)
('NABIL', datetime.date(2024, 12, 3), 587, 58, 55)
('NABIL', datetime.date(2024, 12, 4), 646, 60, 56)
('NABIL', datetime.date(2024, 12, 5), 716, 61, 54)
('NABIL', datetime.date(2024, 12, 8), 839, 64, 53)
('NABIL', datetime.date(2024, 12, 9), 669, 64, 51)
('NABIL', datetime.date(2024, 12, 10), 558, 57, 48)
('NABIL', datetime.date(2024, 12, 11), 686, 61, 52)
('NABIL', datetime.date(2024, 12, 12), 635, 66, 52)


In [4]:

from pathlib import Path
import pandas as pd
tbl = Path("../outputs/tables/sample_hhi_selfloop.csv")
assert tbl.exists(), "run scripts/write_sample_hhi_table.py"
sample = pd.read_csv(tbl)
print(f"rows: {len(sample)}  (NABIL {sample['date'].min()} → {sample['date'].max()})")
display(sample.head(10))
summary = sample[["buyer_hhi", "seller_hhi", "self_loop_volume_ratio",
                  "end_of_day_concentration"]].describe().loc[["mean", "std", "min", "max"]]
display(summary)


rows: 20  (NABIL 2024-12-01 → 2024-12-31)


,date,symbol,total_volume,total_trades,unique_buyers,unique_sellers,buyer_hhi,seller_hhi,self_loop_volume_ratio,end_of_day_concentration,top_3_buyer_share,top_3_seller_share
0,2024-12-01,NABIL,79422.0,695,58,55,0.045248,0.040951,0.020083,0.035758,0.252474,0.237138
1,2024-12-02,NABIL,80685.0,776,59,60,0.047814,0.044765,0.019037,0.038000,0.267807,0.280325
2,2024-12-03,NABIL,62719.0,587,58,55,0.076025,0.060643,0.019771,0.040498,0.409174,0.326647
3,2024-12-04,NABIL,69630.0,646,60,56,0.042193,0.112475,0.010771,0.035358,0.264469,0.420738
4,2024-12-05,NABIL,61452.0,716,61,54,0.037801,0.054844,0.025028,0.037948,0.223313,0.301780
5,2024-12-08,NABIL,76545.0,839,64,53,0.035425,0.037496,0.021334,0.051120,0.204063,0.205957
6,2024-12-09,NABIL,54624.0,669,64,51,0.036709,0.035029,0.037035,0.051607,0.222118,0.167564
7,2024-12-10,NABIL,51966.0,558,57,48,0.033802,0.044107,0.026113,0.067852,0.207155,0.251453
8,2024-12-11,NABIL,70328.0,686,61,52,0.040985,0.037421,0.030699,0.036557,0.255389,0.205381
9,2024-12-12,NABIL,56954.0,635,66,52,0.042286,0.054012,0.020473,0.070987,0.258507,0.314306


,buyer_hhi,seller_hhi,self_loop_volume_ratio,end_of_day_concentration
mean,0.055692,0.098411,0.037598,0.051302
std,0.031107,0.082745,0.070681,0.018870
min,0.031517,0.035029,0.010771,0.031660
max,0.152541,0.325601,0.336512,0.088840


## STATUS REPORT (Phase 1)

| item | value |
|---|---|
| env | python 3.11.16 · venv `nepse-msc` (uv) |
| duckdb | loaded NABIL 2024-12 extract, query executed |
| schema validation | PASS (canonical core columns; broker ids zero-padded) |
| sample metrics | buyer/seller HHI + self-loop + EoD concentration table saved |
| tests | pytest suite green (see `make test-fast`) |
| blockers | none (SV1(b) archive substitution documented in block_log) |
| [] STOP | awaiting user "continue" → Phase 2 |
